# nb_20_silver_clean

Silver, part 1. Takes the raw text tables from bronze and produces typed, cleaned tables. Rows that break a rule are moved to a quarantine table with the reason, so nothing disappears quietly.

What happens to each row
1. Values get real data types (numbers, dates, true/false).
2. Obvious mess is fixed: spaces, upper and lower case in names and emails, phone formats.
3. Each row is checked against the rules for its table. A row that breaks a reject rule goes to `ctl_quarantine` and is not in the silver table.
4. Every rule is counted and written to `ctl_dq_results`.
5. A reconciliation check proves that `bronze rows = silver rows + quarantined rows` for every table.

This notebook rebuilds the silver tables from bronze each time it runs, so running it twice gives the same result. Duplicates (rows the source sent twice) are kept here and removed in part 2.

Output tables: `silver_crm_client`, `silver_pos_transaction`, `silver_ecom_order`, `silver_after_sales_ticket`, `silver_appointment`, and the control tables `ctl_quarantine`, `ctl_dq_results`, `ctl_run_log`.


In [ ]:
# ---- Settings -------------------------------------------------------------------------
# Assumptions about the world that are not in the data files.
DIAL_CODES = {"US": "1", "CA": "1", "FR": "33", "GB": "44", "IT": "39",
              "CH": "41", "KR": "82", "JP": "81", "HK": "852"}
VALID_REGIONS = ["americas", "europe", "asia_pacific"]
VALID_TICKET_STATUS = ["open", "in_progress", "closed"]
VALID_APPOINTMENT_STATUS = ["completed", "cancelled", "no_show"]
VALID_SEGMENTS = ["vic", "priority", "core"]

# Which team should look at a rejected row? (the five failure classes in the design)
RULE_CLASS = {
    "missing_key": "source", "bad_timestamp": "source", "bad_created_at": "source", "bad_closed_at": "source",
    "bad_updated_at": "source", "invalid_quantity": "source", "invalid_is_return": "source",
    "return_sign_mismatch": "source", "missing_price": "source", "invalid_price": "source",
    "future_dated": "source", "invalid_status": "source", "closed_without_date": "source",
    "closed_before_created": "source", "invalid_purchase_region": "source",
    "orphan_client": "integration", "unknown_product": "integration", "unknown_boutique": "integration",
    "unknown_currency": "integration", "unknown_home_boutique": "integration",
    "erased_client": "policy",
}

In [ ]:
# ---- Setup ----------------------------------------------------------------------------
import uuid
from datetime import datetime, timezone
from functools import reduce

import pyspark.sql.functions as F
from pyspark.sql import Row
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, LongType,
                               DoubleType, TimestampType)

SILVER_BATCH_ID = "S" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:6]
print("silver batch id:", SILVER_BATCH_ID)


def now_utc():
    return datetime.now(timezone.utc).replace(tzinfo=None)


dq_rows = []          # becomes ctl_dq_results
run_rows = []         # becomes ctl_run_log
quarantine_dfs = []   # becomes ctl_quarantine
quarantine_counts = {}   # rows sent to quarantine per bronze table, counted independently of the silver table

In [ ]:
# ---- Reference data used by the rules -------------------------------------------------
product_ids = [r["product_id"] for r in spark.table("bronze_product").select("product_id").collect()]
boutique_rows = spark.table("bronze_boutique").select("boutique_id", "country").collect()
boutique_ids = [r["boutique_id"] for r in boutique_rows]
boutique_country = {r["boutique_id"]: r["country"] for r in boutique_rows}
currencies = [r["currency"] for r in spark.table("bronze_fx_rate").select("currency").distinct().collect()]
print(len(product_ids), "products,", len(boutique_ids), "boutiques,", len(currencies), "currencies")

In [ ]:
# ---- The rule engine ------------------------------------------------------------------
# reject_rules: (name, condition that is TRUE when the row is bad). The first rule that fires is the main reason.
# info_rules:   (name, severity, condition). Counted for the quality report. They never remove a row.
def apply_rules(df, table, key_col, reject_rules, info_rules=()):
    started = now_utc()
    raw_cols = [c for c in df.columns if not c.startswith("_") and not c.startswith("t_")]

    for name, cond in reject_rules:
        df = df.withColumn(f"_f_{name}", F.coalesce(cond, F.lit(True)))     # a rule that cannot be evaluated fails the row
    for name, _sev, cond in info_rules:
        df = df.withColumn(f"_f_{name}", F.coalesce(cond, F.lit(False)))
    df = df.cache()

    reject_flags = [F.col(f"_f_{n}") for n, _ in reject_rules]
    failed = reduce(lambda a, b: a | b, reject_flags)
    first_rule = F.coalesce(*[F.when(F.col(f"_f_{n}"), F.lit(n)) for n, _ in reject_rules])
    all_rules = F.concat_ws(",", *[F.when(F.col(f"_f_{n}"), F.lit(n)) for n, _ in reject_rules])
    class_map = F.create_map(*[x for k, v in RULE_CLASS.items() for x in (F.lit(k), F.lit(v))])

    names = ([(n, "reject") for n, _ in reject_rules] + [(n, s) for n, s, _ in info_rules])
    agg = df.agg(F.count("*").alias("_n"),
                 *[F.sum(F.col(f"_f_{n}").cast("int")).alias(n) for n, _ in names]).collect()[0]
    total = int(agg["_n"])
    for n, sev in names:
        bad = int(agg[n] or 0)
        dq_rows.append(Row(batch_id=SILVER_BATCH_ID, layer="silver", table_name=table, rule=n, severity=sev,
                           rows_checked=total, rows_failed=bad,
                           pct_failed=round(100.0 * bad / total, 3) if total else 0.0, checked_at=now_utc()))

    quarantine = (df.filter(failed).select(
        F.lit(SILVER_BATCH_ID).alias("batch_id"), F.lit("silver_clean").alias("stage"),
        F.lit(table).alias("source_table"), F.col("_source_file").alias("source_file"),
        F.when(first_rule == "erased_client", F.sha2(F.col(key_col), 256)).otherwise(F.col(key_col)).alias("record_key"),
        first_rule.alias("rule"), all_rules.alias("all_rules"),
        F.element_at(class_map, first_rule).alias("failure_class"),
        # an erased client's data must not be copied into the quarantine table
        F.when(first_rule == "erased_client", F.lit('{"redacted":"erased_client"}'))
         .otherwise(F.to_json(F.struct(*raw_cols))).alias("raw_record"), F.current_timestamp().alias("quarantined_at")))
    quarantine_counts[table] = quarantine.count()
    quarantine_dfs.append(quarantine)

    clean = df.filter(~failed)
    return clean, total, started


def finish_table(clean, total, out_table, started, bronze_table):
    """Write one silver table and prove that every bronze row is either here or in quarantine."""
    clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(out_table)
    kept = spark.table(out_table).count()
    quarantined = quarantine_counts[bronze_table]          # counted from the quarantine rows, not worked out by subtraction
    gap = total - kept - quarantined
    ok = (gap == 0)
    dq_rows.append(Row(batch_id=SILVER_BATCH_ID, layer="silver", table_name=out_table,
                       rule="reconcile_bronze_to_silver", severity="reconcile", rows_checked=total,
                       rows_failed=abs(gap), pct_failed=round(100.0 * abs(gap) / total, 3) if total else 0.0,
                       checked_at=now_utc()))
    run_rows.append(Row(batch_id=SILVER_BATCH_ID, layer="silver", object_name=out_table,
                        status="success" if ok else "failed", files_loaded=0, rows_loaded=kept,
                        started_at=started, finished_at=now_utc(),
                        failure_class=None if ok else "processing",
                        message=f"bronze={total} silver={kept} quarantined={quarantined} gap={gap}"))
    print(f"{out_table:28s} bronze={total:>7}  silver={kept:>7}  quarantined={quarantined:>5}  gap={gap}")
    return kept, quarantined

In [ ]:
# ---- CRM clients ----------------------------------------------------------------------
# Every row here is one version of a client record. Part 4 turns the versions into history.
crm = spark.table("bronze_crm_client")
country_map = F.create_map(*[x for k, v in boutique_country.items() for x in (F.lit(k), F.lit(v))])
dial_map = F.create_map(*[x for k, v in DIAL_CODES.items() for x in (F.lit(k), F.lit(v))])

def fix_case(c):
    """Names that are all upper or all lower case get a capital first letter. Mixed case is left alone."""
    t = F.trim(c)
    return (F.when(t.isNull(), F.lit(None))
             .when((t == F.upper(t)) | (t == F.lower(t)), F.initcap(t))
             .otherwise(t))

digits = F.regexp_replace(F.col("phone"), r"[^0-9]", "")
country = F.element_at(country_map, F.col("home_boutique_id"))
dial = F.element_at(dial_map, country)

crm = (crm
    .withColumn("t_created", F.to_timestamp("created_at", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("t_updated", F.to_timestamp("updated_at", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("t_first", fix_case(F.col("first_name")))
    .withColumn("t_last", fix_case(F.col("last_name")))
    .withColumn("_email_lc", F.lower(F.trim(F.col("email"))))
    .withColumn("t_email_malformed", (F.col("_email_lc") != "") & ~F.col("_email_lc").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$"))
    .withColumn("t_email", F.when((F.col("_email_lc") == "") | F.col("t_email_malformed"), F.lit(None)).otherwise(F.col("_email_lc")))
    .withColumn("t_country", country)
    .withColumn("t_phone", (
        F.when(F.col("phone").isNull() | (F.trim(F.col("phone")) == ""), F.lit(None))
         .when(F.trim(F.col("phone")).startswith("+") & (F.length(digits) >= 8), F.concat(F.lit("+"), digits))
         .when(digits.startswith("0") & dial.isNotNull(), F.concat(F.lit("+"), dial, F.substring(digits, 2, 20)))
         .when(F.col("t_country").isin("US", "CA") & (F.length(digits) == 10), F.concat(F.lit("+1"), digits))
         .when((F.col("t_country") == "HK") & (F.length(digits) == 8), F.concat(F.lit("+852"), digits))))
    .withColumn("t_share_missing", ~F.col("consent_share_cross_region").isin("0", "1") | F.col("consent_share_cross_region").isNull())
    .withColumn("t_mkt_missing", ~F.col("consent_marketing").isin("0", "1") | F.col("consent_marketing").isNull())
    .withColumn("t_segment", F.when(F.lower(F.trim(F.col("client_segment"))).isin(*VALID_SEGMENTS), F.lower(F.trim(F.col("client_segment")))))
    if "client_segment" in crm.columns else crm.withColumn("t_segment", F.lit(None).cast("string")))

# Clients erased by the retention job (nb_21b) are blocked here, even if an old file is loaded again.
# The tombstone holds only a hash of the client id.
if spark.catalog.tableExists("ctl_erasure_tombstone"):
    erased = spark.table("ctl_erasure_tombstone").select(F.col("client_id_hash").alias("_tomb_hash")).distinct().withColumn("_erased", F.lit(True))
else:
    erased = spark.createDataFrame([], "_tomb_hash string, _erased boolean")
crm = (crm.withColumn("_client_hash", F.sha2(F.col("client_id"), 256))
          .join(erased, F.col("_client_hash") == F.col("_tomb_hash"), "left").drop("_tomb_hash", "_client_hash"))

In [ ]:
started = now_utc()
crm_clean, crm_total, crm_started = apply_rules(
    crm, "bronze_crm_client", "client_id",
    reject_rules=[
        ("erased_client", F.coalesce(F.col("_erased"), F.lit(False))),
        ("missing_key", F.col("client_id").isNull() | (F.trim(F.col("client_id")) == "")),
        ("bad_created_at", F.col("t_created").isNull()),
        ("bad_updated_at", F.col("t_updated").isNull()),
        ("unknown_home_boutique", ~F.col("home_boutique_id").isin(boutique_ids)),
    ],
    info_rules=[
        ("email_missing", "info", F.col("_email_lc").isNull() | (F.col("_email_lc") == "")),
        ("email_malformed", "fix", F.col("t_email_malformed")),
        ("email_trimmed_or_lowercased", "fix", F.col("email").isNotNull() & (F.col("email") != F.col("_email_lc")) & (F.col("_email_lc") != "")),
        ("name_case_fixed", "fix", (F.col("first_name") != F.col("t_first")) | (F.col("last_name") != F.col("t_last"))),
        ("phone_missing", "info", F.col("phone").isNull() | (F.trim(F.col("phone")) == "")),
        ("phone_unparsed", "fix", F.col("phone").isNotNull() & (F.trim(F.col("phone")) != "") & F.col("t_phone").isNull()),
        ("consent_share_missing", "info", F.col("t_share_missing")),
        ("consent_marketing_missing", "info", F.col("t_mkt_missing")),
        ("region_differs_from_folder", "info", F.col("region") != F.col("_region")),
    ])

silver_crm = crm_clean.select(
    F.col("client_id"), F.col("_region").alias("region"),
    F.col("t_first").alias("first_name"), F.col("t_last").alias("last_name"),
    F.col("t_email").alias("email"), F.sha2(F.col("t_email"), 256).alias("email_hash"),
    F.col("t_phone").alias("phone_e164"), F.col("home_boutique_id"), F.col("t_country").alias("home_country"),
    F.col("t_created").alias("created_at"), F.col("t_updated").alias("updated_at"),
    # Missing consent is treated as NO consent. The flag keeps the fact that it was missing.
    F.when(F.col("consent_share_cross_region") == "1", F.lit(True)).otherwise(F.lit(False)).alias("consent_share_cross_region"),
    F.col("t_share_missing").alias("consent_share_missing"),
    F.when(F.col("consent_marketing") == "1", F.lit(True)).otherwise(F.lit(False)).alias("consent_marketing"),
    F.col("t_mkt_missing").alias("consent_marketing_missing"),
    F.col("t_segment").alias("client_segment"),
    F.col("_source_file").alias("source_file"), F.to_date("_dt").alias("file_date"),
    F.col("_batch_id").alias("bronze_batch_id"), F.lit(SILVER_BATCH_ID).alias("silver_batch_id"))

finish_table(silver_crm, crm_total, "silver_crm_client", crm_started, "bronze_crm_client")
known_clients = spark.table("silver_crm_client").select("client_id").distinct().withColumn("_known_client", F.lit(True))

In [ ]:
# ---- Sales: boutique (pos) and online (ecom) ------------------------------------------
def prep_sales(bronze_name, ts_col):
    b = spark.table(bronze_name)
    return (b
        .withColumn("t_qty", F.col("quantity").cast("int"))
        .withColumn("t_price", F.col("unit_price").cast("decimal(18,2)"))
        .withColumn("t_ts", F.to_timestamp(F.col(ts_col), "yyyy-MM-dd HH:mm:ss"))
        .withColumn("t_is_return", F.col("is_return").cast("int"))
        .join(known_clients, "client_id", "left"))


def sales_rules(key_col, with_boutique):
    rules = [
        ("missing_key", F.col(key_col).isNull() | (F.trim(F.col(key_col)) == "")),
        ("bad_timestamp", F.col("t_ts").isNull()),
        ("invalid_quantity", F.col("t_qty").isNull() | (F.col("t_qty") == 0)),
        ("invalid_is_return", ~F.col("t_is_return").isin(0, 1)),
        ("return_sign_mismatch", ((F.col("t_is_return") == 1) & (F.col("t_qty") > 0)) | ((F.col("t_is_return") == 0) & (F.col("t_qty") < 0))),
        ("orphan_client", F.col("_known_client").isNull()),
        ("unknown_product", ~F.col("product_id").isin(product_ids)),
        ("missing_price", F.col("unit_price").isNull() | (F.trim(F.col("unit_price")) == "")),
        ("invalid_price", F.col("unit_price").isNotNull() & (F.trim(F.col("unit_price")) != "") & (F.col("t_price").isNull() | (F.col("t_price") <= 0))),
        ("future_dated", F.to_date("t_ts") > F.to_date("_dt")),
        ("unknown_currency", ~F.col("currency").isin(currencies)),
    ]
    if with_boutique:
        rules.append(("unknown_boutique", ~F.col("boutique_id").isin(boutique_ids)))
    return rules


def sales_output(clean, key_col, ts_name, with_boutique, extra_cols):
    cols = [F.col(key_col), F.col("client_id")]
    if with_boutique:
        cols.append(F.col("boutique_id"))
    cols += [F.col("product_id"), F.col("t_qty").alias("quantity"), F.col("t_price").alias("unit_price"),
             F.col("currency"), (F.col("t_is_return") == 1).alias("is_return"),
             F.col("t_ts").alias(ts_name), F.to_date("t_ts").alias("event_date"),
             F.col("_region").alias("region")]
    cols += [F.col(c) for c in extra_cols]
    cols += [F.col("_source_file").alias("source_file"), F.to_date("_dt").alias("file_date"),
             F.col("_batch_id").alias("bronze_batch_id"), F.lit(SILVER_BATCH_ID).alias("silver_batch_id")]
    return clean.select(*cols)


# boutique sales
pos = prep_sales("bronze_pos_transaction", "transaction_ts")
pos_clean, pos_total, pos_started = apply_rules(
    pos, "bronze_pos_transaction", "transaction_id", sales_rules("transaction_id", True),
    info_rules=[("return_rows", "info", F.col("t_is_return") == 1)])
extra = [c for c in ["sales_associate_id"] if c in pos.columns]
finish_table(sales_output(pos_clean, "transaction_id", "transaction_ts", True, extra),
             pos_total, "silver_pos_transaction", pos_started, "bronze_pos_transaction")

# online orders
ecom = prep_sales("bronze_ecom_order", "order_ts")
ecom_clean, ecom_total, ecom_started = apply_rules(
    ecom, "bronze_ecom_order", "order_id", sales_rules("order_id", False),
    info_rules=[("return_rows", "info", F.col("t_is_return") == 1)])
finish_table(sales_output(ecom_clean, "order_id", "order_ts", False, []),
             ecom_total, "silver_ecom_order", ecom_started, "bronze_ecom_order")

In [ ]:
# ---- Repairs and services -------------------------------------------------------------
tk = (spark.table("bronze_after_sales_ticket")
      .withColumn("t_created", F.to_timestamp("created_at", "yyyy-MM-dd HH:mm:ss"))
      .withColumn("t_closed", F.to_timestamp("closed_at", "yyyy-MM-dd HH:mm:ss"))
      .join(known_clients, "client_id", "left"))
has_closed = F.col("closed_at").isNotNull() & (F.trim(F.col("closed_at")) != "")
tk_clean, tk_total, tk_started = apply_rules(
    tk, "bronze_after_sales_ticket", "ticket_id",
    reject_rules=[
        ("missing_key", F.col("ticket_id").isNull() | (F.trim(F.col("ticket_id")) == "")),
        ("bad_created_at", F.col("t_created").isNull()),
        ("bad_closed_at", has_closed & F.col("t_closed").isNull()),
        ("invalid_status", ~F.col("status").isin(VALID_TICKET_STATUS)),
        ("closed_without_date", (F.col("status") == "closed") & ~has_closed),
        ("closed_before_created", F.col("t_closed").isNotNull() & (F.col("t_closed") < F.col("t_created"))),
        ("future_dated", F.to_date("t_created") > F.to_date("_dt")),
        ("orphan_client", F.col("_known_client").isNull()),
        ("unknown_product", ~F.col("product_id").isin(product_ids)),
        ("unknown_boutique", ~F.col("boutique_id").isin(boutique_ids)),
        ("invalid_purchase_region", ~F.col("purchase_region").isin(VALID_REGIONS)),
    ],
    info_rules=[("repaired_outside_purchase_region", "info", F.col("purchase_region") != F.col("_region"))])

silver_tk = tk_clean.select(
    "ticket_id", "client_id", "boutique_id", "product_id", "status", "reason",
    F.col("t_created").alias("created_at"), F.col("t_closed").alias("closed_at"),
    F.col("purchase_region"), F.col("_region").alias("service_region"),
    (F.col("purchase_region") != F.col("_region")).alias("is_cross_region_service"),
    F.col("_source_file").alias("source_file"), F.to_date("_dt").alias("file_date"),
    F.col("_batch_id").alias("bronze_batch_id"), F.lit(SILVER_BATCH_ID).alias("silver_batch_id"))
finish_table(silver_tk, tk_total, "silver_after_sales_ticket", tk_started, "bronze_after_sales_ticket")

# ---- Appointments ---------------------------------------------------------------------
ap = (spark.table("bronze_appointment")
      .withColumn("t_ts", F.to_timestamp("appointment_ts", "yyyy-MM-dd HH:mm:ss"))
      .join(known_clients, "client_id", "left"))
ap_clean, ap_total, ap_started = apply_rules(
    ap, "bronze_appointment", "appointment_id",
    reject_rules=[
        ("missing_key", F.col("appointment_id").isNull() | (F.trim(F.col("appointment_id")) == "")),
        ("bad_timestamp", F.col("t_ts").isNull()),
        ("invalid_status", ~F.col("status").isin(VALID_APPOINTMENT_STATUS)),
        ("orphan_client", F.col("_known_client").isNull()),
        ("unknown_boutique", ~F.col("boutique_id").isin(boutique_ids)),
    ])
silver_ap = ap_clean.select(
    "appointment_id", "client_id", "boutique_id", "advisor_id", F.col("t_ts").alias("appointment_ts"),
    "status", F.col("_region").alias("region"),
    F.col("_source_file").alias("source_file"), F.to_date("_dt").alias("file_date"),
    F.col("_batch_id").alias("bronze_batch_id"), F.lit(SILVER_BATCH_ID).alias("silver_batch_id"))
finish_table(silver_ap, ap_total, "silver_appointment", ap_started, "bronze_appointment")

In [ ]:
# ---- Write the control tables ---------------------------------------------------------
QUARANTINE_COLS = ["batch_id", "stage", "source_table", "source_file", "record_key", "rule",
                   "all_rules", "failure_class", "raw_record", "quarantined_at"]
q = reduce(lambda a, b: a.unionByName(b), quarantine_dfs).select(*QUARANTINE_COLS)
q.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("ctl_quarantine")

DQ_SCHEMA = StructType([
    StructField("batch_id", StringType()), StructField("layer", StringType()),
    StructField("table_name", StringType()), StructField("rule", StringType()),
    StructField("severity", StringType()), StructField("rows_checked", LongType()),
    StructField("rows_failed", LongType()), StructField("pct_failed", DoubleType()),
    StructField("checked_at", TimestampType())])
RUN_SCHEMA = StructType([
    StructField("batch_id", StringType()), StructField("layer", StringType()),
    StructField("object_name", StringType()), StructField("status", StringType()),
    StructField("files_loaded", IntegerType()), StructField("rows_loaded", LongType()),
    StructField("started_at", TimestampType()), StructField("finished_at", TimestampType()),
    StructField("failure_class", StringType()), StructField("message", StringType())])

spark.createDataFrame(dq_rows, schema=DQ_SCHEMA).write.format("delta").mode("append") \
     .option("mergeSchema", "true").saveAsTable("ctl_dq_results")
spark.createDataFrame(run_rows, schema=RUN_SCHEMA).write.format("delta").mode("append") \
     .option("mergeSchema", "true").saveAsTable("ctl_run_log")

bad = [r for r in run_rows if r.status != "success"]
if bad:
    raise RuntimeError("Silver reconciliation failed for: " + ", ".join(r.object_name for r in bad))
print("silver part 1 finished, batch", SILVER_BATCH_ID)

## Check the result
Read-only cells. Run them after the load.

In [ ]:
# Reject rules that fired in this run, most common first
(spark.table("ctl_dq_results").filter(F.col("batch_id") == SILVER_BATCH_ID)
      .filter("severity = 'reject' and rows_failed > 0")
      .select("table_name", "rule", "rows_failed", "pct_failed")
      .orderBy(F.col("rows_failed").desc()).show(40, truncate=False))

In [ ]:
# Cleaning fixes and facts found in the data (these rows are kept)
(spark.table("ctl_dq_results").filter(F.col("batch_id") == SILVER_BATCH_ID)
      .filter("severity in ('fix','info') and rows_failed > 0")
      .select("table_name", "rule", "severity", "rows_failed", "rows_checked")
      .orderBy("table_name", "rule").show(60, truncate=False))

In [ ]:
# Does every bronze row end up in silver or in quarantine?
rows = []
for bronze, silver in [("bronze_crm_client", "silver_crm_client"), ("bronze_pos_transaction", "silver_pos_transaction"),
                       ("bronze_ecom_order", "silver_ecom_order"), ("bronze_after_sales_ticket", "silver_after_sales_ticket"),
                       ("bronze_appointment", "silver_appointment")]:
    b = spark.table(bronze).count()
    s = spark.table(silver).count()
    qn = spark.table("ctl_quarantine").filter((F.col("batch_id") == SILVER_BATCH_ID) & (F.col("source_table") == bronze)).count()
    rows.append((bronze, b, s, qn, b - s - qn))
spark.createDataFrame(rows, ["bronze_table", "bronze_rows", "silver_rows", "quarantined", "difference"]).show(truncate=False)

In [ ]:
# Rejected rows counted by distinct record. A record the source sent twice is rejected twice
# (the copy is removed in part 2), so these numbers can be a little lower than the rule counts above.
(spark.table("ctl_quarantine").filter(F.col("batch_id") == SILVER_BATCH_ID)
      .groupBy("rule").agg(F.count("*").alias("rejected_rows"), F.countDistinct("record_key").alias("distinct_records"))
      .orderBy(F.col("rejected_rows").desc()).show(20, truncate=False))

In [ ]:
# A few quarantined rows, to see what they look like
(spark.table("ctl_quarantine").filter(F.col("batch_id") == SILVER_BATCH_ID)
      .select("source_table", "record_key", "rule", "failure_class", "raw_record")
      .show(8, truncate=80))